Grupo 02 - D2 NYC Taxi

# Bloco 01 | Escolha do dataset

**Dataset escolhido:** D2 - NYC Táxi amarelo, janeiro de 2024, mais a tabela de zonas.

Fontes:
- `yellow_tripdata_2024-01.parquet` (viagens)
- `taxi_zone_lookup.csv` (zonas)

**P1. Qual dataset o grupo escolheu e por que ele foi preferido ao segundo candidato?**

Escolhemos o D2 (NYC Táxi). Ele atende ao critério de volume e qualidade: uma fonte com 2.964.624 linhas (muito mais que as 10 mil exigidas) e uma segunda fonte (zonas) que precisa ser combinada. Os dois arquivos baixam sem cadastro, e a coluna de data de embarque permite agregar por hora.
Preferimos o D2 (NYC Táxi) ao D4 (Banco Central: Selic, IPCA e dólar) porque o volume de dados é consideravelmente maior, deixando o tratamento e a análise mais interessantes.

**P2. Quantas fontes ele possui e quantas linhas tem a maior delas?**

O dataset tem **2 fontes**: o arquivo de viagens (parquet) e o arquivo de zonas (CSV). A maior é a de viagens, com **2.964.624 linhas** e 19 colunas. A de zonas tem 265 linhas e 4 colunas.

**P3. Que problema de qualidade vocês já enxergaram apenas olhando uma amostra?**

Olhando as amostras e rodando contagens simples, encontramos vários problemas na tabela de viagens (2.964.624 linhas) e um na de zonas:

1. **Nulos em `passenger_count`:** 140.162 viagens (cerca de 4,7%) não têm a quantidade de passageiros. Isso também explica por que a coluna vem como `float64` em vez de inteiro.
2. **Valores monetários inválidos:** 35.504 viagens (cerca de 1,2%) têm `total_amount` negativo e 416 têm `total_amount` igual a zero. Uma receita negativa não faz sentido para uma corrida normal e distorceria a soma por zona e hora.
3. **Distância zero:** 60.371 viagens (cerca de 2,0%) têm `trip_distance` igual a zero, e 31.465 têm `passenger_count` igual a zero. São registros suspeitos, e ainda não sabemos se são corridas canceladas ou erros de registro.
4. **Datas fora do período:** 18 embarques estão fora de janeiro de 2024, em um arquivo que deveria conter só esse mês.
5. **Nulos na tabela de zonas:** `Borough` tem 1 nulo, `Zone` tem 1 e `service_zone` tem 2, em 265 linhas.

# BLOCO 2 | Conhecer a fonte e definir a pergunta

In [1]:
import os
from pathlib import Path

print("Trabalhando em:", os.getcwd())
print("Pasta 'fonte' existe?", Path("fonte").exists())

Trabalhando em: c:\desenvolvimento-faculdade\I.A-C.D-Grupo-2-2-periodo--Faculdade-Donaduzzi\pipeline_de_dados
Pasta 'fonte' existe? True


In [5]:
import pandas as pd
from pathlib import Path

PASTA_FONTE = Path("fonte")

# Fonte 1: viagens
df_viagens = pd.read_parquet(PASTA_FONTE / 'yellow_tripdata_2024-01.parquet')

print("shape:", df_viagens.shape)
print()
display(df_viagens.head())
print()
df_viagens.info(show_counts=True)

shape: (2964624, 19)



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0



<class 'pandas.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               2964624 non-null  int32         
 1   tpep_pickup_datetime   2964624 non-null  datetime64[us]
 2   tpep_dropoff_datetime  2964624 non-null  datetime64[us]
 3   passenger_count        2824462 non-null  float64       
 4   trip_distance          2964624 non-null  float64       
 5   RatecodeID             2824462 non-null  float64       
 6   store_and_fwd_flag     2824462 non-null  str           
 7   PULocationID           2964624 non-null  int32         
 8   DOLocationID           2964624 non-null  int32         
 9   payment_type           2964624 non-null  int64         
 10  fare_amount            2964624 non-null  float64       
 11  extra                  2964624 non-null  float64       
 12  mta_tax                2964624 non-nul

In [6]:
# Contagem de nulos por coluna (viagens)
nulos_viagens = df_viagens.isna().sum()
display(nulos_viagens.to_frame("nulos").assign(pct=(nulos_viagens / len(df_viagens) * 100).round(2)))

,nulos,pct
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,140162,4.73
trip_distance,0,0.00
RatecodeID,140162,4.73
store_and_fwd_flag,140162,4.73
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00


In [8]:
# Fonte 2: zonas
df_zonas = pd.read_csv(PASTA_FONTE /'taxi_zone_lookup.csv')

print("shape:", df_zonas.shape)
print()
display(df_zonas.head())
print()
df_zonas.info()
print()
print("Nulos por coluna:")
display(df_zonas.isna().sum())

shape: (265, 4)



,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone



<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   LocationID    265 non-null    int64
 1   Borough       264 non-null    str  
 2   Zone          264 non-null    str  
 3   service_zone  263 non-null    str  
dtypes: int64(1), str(3)
memory usage: 17.0 KB

Nulos por coluna:


LocationID      0
Borough         1
Zone            1
service_zone    2
dtype: int64

In [12]:
# Os nulos das zonas são nulos de verdade? Olhem as linhas com nulo.
# Dica: o pandas converte os textos "NA" e "N/A" em NaN automaticamente.
display(df_zonas[df_zonas.isna().any(axis=1)])

# Compare que a leitura que NÃO converte texto em nulo:
df_zonas_raw = pd.read_csv(PASTA_FONTE / 'taxi_zone_lookup.csv', keep_default_na=False)
display(df_zonas_raw[df_zonas_raw['LocationID'].isin(df_zonas[df_zonas.isna().any(axis=1)]['LocationID'])])

,LocationID,Borough,Zone,service_zone
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


,LocationID,Borough,Zone,service_zone
263,264,Unknown,N/A,N/A
264,265,N/A,Outside of NYC,N/A


In [15]:
# Chave de ligação e coluna de data
print("Chave: viagens.PULocationID  ->  zonas.LocationID")
print("IDs de embarque que não existem na tabela de zonas:",
      (~df_viagens['PULocationID'].isin(df_zonas['LocationID'])).sum())

# Coluna de data usada para agregar por hora
print()
print("Primeiro embarque:", df_viagens['tpep_pickup_datetime'].min())
print("Último embarque:  ", df_viagens['tpep_pickup_datetime'].max())

Chave: viagens.PULocationID  ->  zonas.LocationID
IDs de embarque que não existem na tabela de zonas: 0

Primeiro embarque: 2002-12-31 22:59:39
Último embarque:   2024-02-01 00:01:15


In [16]:
# Problemas de qualidade visíveis (usar esses números na P3 do Bloco 1)
inicio, fim = pd.Timestamp('2024-01-01'), pd.Timestamp('2024-02-01')
fora_periodo = ~df_viagens['tpep_pickup_datetime'].between(inicio, fim, inclusive='left')

print("Embarques fora de jan/2024:    ", fora_periodo.sum())
print("total_amount negativo:         ", (df_viagens['total_amount'] < 0).sum())
print("total_amount igual a zero:     ", (df_viagens['total_amount'] == 0).sum())
print("trip_distance igual a zero:    ", (df_viagens['trip_distance'] == 0).sum())
print("passenger_count nulo:          ", df_viagens['passenger_count'].isna().sum())
print("passenger_count igual a zero:  ", (df_viagens['passenger_count'] == 0).sum())

Embarques fora de jan/2024:     18
total_amount negativo:          35504
total_amount igual a zero:      416
trip_distance igual a zero:     60371
passenger_count nulo:           140162
passenger_count igual a zero:   31465


### Respostas do Bloco 2

**P1. Que tabela agregada responde à pergunta do grupo? Nessa tabela, uma linha representa o quê?**

A tabela é a **receita por zona de embarque e hora do dia** (`receita_zona_hora`). Cada linha representa **uma zona de embarque em uma hora do dia** (por exemplo, a zona X às 18h), com a receita total e, pelo menos, uma segunda métrica (número de corridas, por exemplo).

**P2. Quais fontes ou colunas NÃO são necessárias para a pergunta? Por que deixá-las de fora é uma decisão da etapa de Extract?**

Das 19 colunas de viagens, a pergunta só precisa de: data/hora de embarque (`tpep_pickup_datetime`), zona de embarque (`PULocationID`) e valores (`fare_amount`, `total_amount`). Mantivemos também `trip_distance` para uma possível segunda pergunta (Bloco 9).
Ficam de fora, por exemplo: `tpep_dropoff_datetime`, `DOLocationID`, `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `payment_type`, e as taxas e sobretaxas.
Deixar de fora é decisão do Extract porque é nessa etapa que escolhemos o que entra no pipeline: ler só as colunas necessárias reduz memória, tempo e tamanho da Bronze, e o que não entra na Bronze não precisa ser tratado nas camadas seguintes.

**P3. Algum nulo encontrado pode estar carregando informação em vez de indicar erro? Se sim, qual é o nulo e o que ele significa?**

Sim. Na tabela de zonas, os nulos de `Borough`, `Zone` e `service_zone` não são falta de dado: são textos que o pandas converteu em nulo (`NA`, `N/A`). Eles aparecem nas linhas de `LocationID` 264 e 265, que representam corridas com zona **desconhecida** ou fora da cidade. Ou seja, o nulo significa "zona não identificada", e isso é uma informação.
_[Confiram na célula acima quais linhas apareceram e ajustem o texto com os valores exatos.]_

# BLOCO 3 | Extract: construir a Bronze
Regras da Bronze: **cópia fiel da fonte**, só com as colunas necessárias e com metadados de controle. Nenhuma correção.

In [24]:
from pathlib import Path
from datetime import datetime
import urllib.request

# Configuração (no Bloco 10 vocês vão ampliar este dicionário)
CONFIG = {
    "bronze": "data/bronze",
    "silver": "data/silver",
    "gold":   "data/gold",
    "fontes": {
        "viagens": {
            "arquivo_local": "fonte/yellow_tripdata_2024-01.parquet",
            "url": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
            "colunas": ["tpep_pickup_datetime", "PULocationID",
                        "fare_amount", "total_amount", "trip_distance"],
        },
        "zonas": {
            "arquivo_local": "fonte/taxi_zone_lookup.csv",
            "url": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
        },
    },
}

# Passo 1: criar as pastas das três camadas
for camada in ("bronze", "silver", "gold"):
    Path(CONFIG[camada]).mkdir(parents=True, exist_ok=True)
print("Pastas criadas:", [CONFIG[c] for c in ("bronze", "silver", "gold")])

Pastas criadas: ['data/bronze', 'data/silver', 'data/gold']


In [25]:
def _obter_arquivo(fonte):
    """Usa o arquivo da pasta 'fonte'. Se não achar, para com uma mensagem clara."""
    caminho = Path(fonte["arquivo_local"])
    if not caminho.exists():
        raise FileNotFoundError(f"Não achei {caminho}. Confira se a pasta 'fonte' veio do Git.")
    return caminho


def extract(destino_bronze):
    """Lê cada fonte necessária e grava na Bronze SEM corrigir nada.
    Acrescenta metadados de controle: _data_extracao e _origem."""
    destino = Path(destino_bronze)
    destino.mkdir(parents=True, exist_ok=True)
    agora = datetime.now().isoformat(timespec="seconds")

    # Fonte em arquivo (parquet): ler só as colunas necessárias
    f = CONFIG["fontes"]["viagens"]
    viagens = pd.read_parquet(_obter_arquivo(f), columns=f["colunas"])
    viagens["_data_extracao"] = agora
    viagens["_origem"] = f["url"]
    viagens.to_parquet(destino / "viagens.parquet", index=False)   # sobrescreve: não duplica

    # Fonte em arquivo (csv): keep_default_na=False preserva "NA" e "N/A" como texto, sem converter em nulo
    f = CONFIG["fontes"]["zonas"]
    zonas = pd.read_csv(_obter_arquivo(f), keep_default_na=False)
    zonas["_data_extracao"] = agora
    zonas["_origem"] = f["url"]
    zonas.to_parquet(destino / "zonas.parquet", index=False)

    return {"viagens": len(viagens), "zonas": len(zonas)}

In [26]:
def linhas_bronze():
    pasta = Path(CONFIG["bronze"])
    return {arq.stem: pd.read_parquet(arq).shape for arq in sorted(pasta.glob("*.parquet"))}

# Executar extract() duas vezes seguidas e anotar as linhas após cada execução
extract(CONFIG["bronze"])
print("Após a 1ª execução:", linhas_bronze())

extract(CONFIG["bronze"])
print("Após a 2ª execução:", linhas_bronze())

Após a 1ª execução: {'viagens': (2964624, 7), 'zonas': (265, 6)}
Após a 2ª execução: {'viagens': (2964624, 7), 'zonas': (265, 6)}


In [20]:
# Conferir a Bronze com a fonte original
bronze_viagens = pd.read_parquet(Path(CONFIG["bronze"]) / "viagens.parquet")
bronze_zonas = pd.read_parquet(Path(CONFIG["bronze"]) / "zonas.parquet")

print("Viagens - fonte:", df_viagens.shape[0], "linhas | Bronze:", bronze_viagens.shape)
print("Zonas   - fonte:", df_zonas.shape[0], "linhas | Bronze:", bronze_zonas.shape)
print()
display(bronze_viagens.head(3))
display(bronze_zonas.tail(3))

Viagens - fonte: 2964624 linhas | Bronze: (2964624, 7)
Zonas   - fonte: 265 linhas | Bronze: (265, 6)



,tpep_pickup_datetime,PULocationID,fare_amount,total_amount,trip_distance,_data_extracao,_origem
0,2024-01-01 00:57:55,186,17.7,22.70,1.72,2026-10-22T00:03:38,https://d37ci6vzurychx.cloudfront.net/trip-dat...
1,2024-01-01 00:03:00,140,10.0,18.75,1.80,2026-10-22T00:03:38,https://d37ci6vzurychx.cloudfront.net/trip-dat...
2,2024-01-01 00:17:06,236,23.3,31.30,4.70,2026-10-22T00:03:38,https://d37ci6vzurychx.cloudfront.net/trip-dat...


,LocationID,Borough,Zone,service_zone,_data_extracao,_origem
262,263,Manhattan,Yorkville West,Yellow Zone,2026-10-22T00:03:38,https://d37ci6vzurychx.cloudfront.net/misc/tax...
263,264,Unknown,N/A,N/A,2026-10-22T00:03:38,https://d37ci6vzurychx.cloudfront.net/misc/tax...
264,265,N/A,Outside of NYC,N/A,2026-10-22T00:03:38,https://d37ci6vzurychx.cloudfront.net/misc/tax...


### Respostas do Bloco 3

**P1. Quantas linhas e colunas tem cada arquivo da Bronze? Os números batem com os da fonte original?**

- `viagens.parquet`: **2.964.624 linhas** e **7 colunas** (as 5 colunas escolhidas + `_data_extracao` + `_origem`). A fonte original tem 2.964.624 linhas e 19 colunas.
- `zonas.parquet`: **265 linhas** e **6 colunas** (as 4 originais + os 2 metadados). A fonte tem 265 linhas e 4 colunas.

As linhas batem com a fonte nos dois arquivos. As colunas diferem de propósito: em viagens deixamos de fora 14 colunas (decisão do Extract) e nos dois arquivos acrescentamos os 2 metadados.
_[Confiram os números de colunas na saída da célula de conferência.]_

**P2. O que aconteceu com o número de linhas na segunda execução? Se mudou, por que mudou e como corrigir?**

O número de linhas **não mudou** (2.964.624 em viagens e 265 em zonas nas duas execuções), porque `extract()` grava cada arquivo com `to_parquet`, que **sobrescreve** o arquivo. Se tivéssemos acrescentado (append) em vez de sobrescrever, a segunda execução dobraria as linhas (5.929.248 em viagens). A correção seria sobrescrever a Bronze a cada execução, ou controlar duplicidade por uma chave.

**P3. Por que a Bronze não deve receber nenhuma correção, mesmo que o grupo já conheça os problemas do dado?**

Porque a Bronze é o registro fiel do que a fonte entregou. Se corrigirmos ali, perdemos o dado original: não conseguimos auditar o que foi alterado, nem refazer a limpeza caso uma regra mude. Com a Bronze intacta, qualquer correção fica na Silver, onde pode ser mudada e reprocessada sem baixar a fonte de novo. Por isso lemos as zonas com `keep_default_na=False`: assim `NA` e `N/A` continuam como texto, exatamente como vieram.

**P4. Se a fonte sair do ar amanhã, o que do pipeline continua funcionando?**

Tudo o que parte da Bronze: `transform()` lê só da Bronze e `load()` lê só da Silver, então Silver e Gold podem ser reconstruídas normalmente. Só o `extract()` deixa de funcionar (e a carga de dados novos), porque ele depende da fonte.

# Bloco 04 |

# Bloco 05 |

# Bloco 06 |

# Bloco 07 |

# Bloco 08 |

# Bloco 09 |

# Bloco 10 |

# Bloco 11 |